# A3.5 · Tool permission models

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.4 · MCP is not a security boundary](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**.

| | |
|---|---|
| Open-source tooling | kmcp, OPA |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A3.1 used a tool policy without examining it. This lesson is about the policy
itself, because it is where the autonomy ladder stops being vocabulary and
becomes a config file someone edits.

Three distinct verbs, and the difference matters:

- **allow** — the agent may call this freely. Everything here is in the L2.5
  blast-radius budget.
- **require_approval** — a human must confirm. This is L2, and its value decays
  with volume (A1.2 measured that).
- **deny** — the agent may never call this, *even with approval*. This is the
  boundary that makes L2.5 meaningful.

And one property that matters more than all three: **deny-by-default**. A tool
nobody listed must be refused. This is what makes the policy hold when someone
adds a capability and forgets the policy file — which is the normal case, not
the exception.

## 2 · Demo — the same agent under four permission models

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ToolPolicy:
    allow: set = field(default_factory=set)
    require_approval: set = field(default_factory=set)
    deny: set = field(default_factory=set)

    def check(self, tool, approved=False):
        if tool in self.deny:
            return False, "denied outright (not available at any approval level)"
        if tool in self.require_approval:
            return (True, "approved by a human") if approved else \
                   (False, "requires approval, none presented")
        if tool in self.allow:
            return True, "on the allowlist"
        return False, "not on the allowlist (deny by default)"

TOOLS = ["read_file", "search_code", "write_file", "open_pr", "merge_pr",
         "deploy_prod", "rotate_credential", "exec_python"]

MODELS = {
 "M0 · no policy (framework default)":
   ToolPolicy(allow=set(TOOLS)),
 "M1 · L2 — approve every writer":
   ToolPolicy(allow={"read_file", "search_code"},
              require_approval={"write_file", "open_pr", "merge_pr",
                                "deploy_prod", "rotate_credential", "exec_python"}),
 "M2 · L2.5 — bounded set, some tools never":
   ToolPolicy(allow={"read_file", "search_code", "write_file", "open_pr"},
              require_approval={"merge_pr"},
              deny={"deploy_prod", "rotate_credential", "exec_python"}),
 "M3 · L1 — read only":
   ToolPolicy(allow={"read_file", "search_code"}),
}
for name, pol in MODELS.items():
    print(name)
    for t in TOOLS:
        ok_no,  _ = pol.check(t, approved=False)
        ok_yes, _ = pol.check(t, approved=True)
        state = ("allow" if ok_no else "approval" if ok_yes else "deny")
        print(f"   {t:20s}{state}")
    print()

## 3 · Where it breaks — the tool nobody added to the policy

Six weeks after launch someone adds a tool. The policy file is in a different repository, owned by a different team, and is not updated. What happens next is decided entirely by the default.

In [ ]:
NEW_TOOL = "run_terraform"          # added to the agent, not to the policy

print("a new tool appears and nobody updated the policy:")
for name, pol in MODELS.items():
    ok, why = pol.check(NEW_TOOL, approved=False)
    verdict = "AVAILABLE — unreviewed" if ok else "refused"
    print(f"   {name:38s} {verdict}")
    if ok:
        print(f"      → {why}")

print("\nM0 permits it because its allowlist was 'everything known at the time'.")
print("The other three refuse it, not because anyone anticipated run_terraform,")
print("but because they refuse anything unlisted. That is the whole property.")

## 4 · The control — deny-by-default, plus a budget

Deny-by-default stops the unknown tool. It does not stop someone deliberately adding a wide tool to the allow list. For that you need the A1.4 budget, checked against the policy.

In [ ]:
SCOPE = {"read_file": ("self", True), "search_code": ("self", True),
         "write_file": ("project", True), "open_pr": ("project", True),
         "merge_pr": ("project", False), "deploy_prod": ("org", False),
         "rotate_credential": ("org", False), "exec_python": ("tenant", False),
         "run_terraform": ("org", False)}
WEIGHT = {"self": 0, "project": 3, "tenant": 8, "org": 20}
BUDGET = {"L1": 0, "L2": 0, "L2.5": 12, "L3": 60}

def policy_blast(pol):
    total = 0
    for tool in pol.allow:                     # gated + denied tools score zero
        scope, rev = SCOPE.get(tool, ("self", True))
        total += WEIGHT[scope] * (1 if rev else 2)
    return total

def review(name, pol, rung):
    b = policy_blast(pol)
    ok = b <= BUDGET[rung]
    print(f"{'PASS' if ok else 'FAIL'}  {name:38s} blast={b:3d} budget={BUDGET[rung]:3d} ({rung})")
    return ok

review("M2 · L2.5",              MODELS["M2 · L2.5 — bounded set, some tools never"], "L2.5")
review("M0 · no policy",         MODELS["M0 · no policy (framework default)"],        "L2.5")
review("M3 · L1 read only",      MODELS["M3 · L1 — read only"],                       "L1")

# someone moves merge_pr from require_approval to allow "to speed things up"
loosened = ToolPolicy(allow={"read_file", "search_code", "write_file",
                             "open_pr", "merge_pr"},
                      deny={"deploy_prod", "rotate_credential", "exec_python"})
review("M2 loosened (merge_pr allowed)", loosened, "L2.5")

In [ ]:
# Verify: exhaustively, no policy may permit a denied tool at any approval level.
failures = []
for name, pol in list(MODELS.items()) + [("loosened", loosened)]:
    for tool in TOOLS + [NEW_TOOL]:
        for approved in (False, True):
            ok, _ = pol.check(tool, approved)
            if tool in pol.deny and ok:
                failures.append((name, tool, approved))
            if ok and tool not in pol.allow and tool not in pol.require_approval:
                failures.append((name, tool, approved))
print(f"policies checked: {len(MODELS)+1} · tools: {len(TOOLS)+1} · violations: {len(failures)}")
assert not failures
print("Invariant holds: deny is absolute, and nothing unlisted is ever permitted.")

## What you just proved

The four models show the same tools in different states. The unlisted `run_terraform` is available and unreviewed only under the no-policy model. The budget check passes M2 (blast 6) and the read-only model, and fails the no-policy model and the loosened variant where `merge_pr` was promoted to allow. The exhaustive check reports zero violations.

## Your turn

Measure the median time between a human approval request and the approval for one of your agents. Under two seconds means the gate is a click-through, and those tools should move to `deny` with a separate narrower agent owning them.

---

**Next → [A3.6 · Runtime containment levers](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*